# House Prices - Advanced Regression Techniques

## Notebook 03: Modeling and Optimization

This notebook trains, evaluates, compares, and optimizes regression models for the Kaggle House Prices competition.

The feature-engineered datasets from `02_feature_engineering.ipynb` are used as input.

The main goals are:

1. Load the processed train and test matrices.
2. Define the competition-aligned evaluation metric.
3. Train multiple regression models.
4. Compare models using cross-validation.
5. Tune strong candidate models.
6. Build a final blended model.
7. Generate predictions for the Kaggle test set.
8. Create a valid `submission.csv` file.

In [1]:
# ============================================================
# 1. Imports and Configuration
# ============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", message="X does not have valid feature names.*")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

RANDOM_STATE = 42

---
## 1. Load Feature-Engineered Data

The previous notebook saved the processed training matrix, processed test matrix, log-transformed target, and test identifiers.

The modeling notebook uses:

- `X_train_fe.csv`
- `X_test_fe.csv`
- `y_train_log.csv`
- `test_ids.csv`

The target is already transformed as:

```python
y = np.log1p(SalePrice)
```

Therefore, model predictions must later be converted back using:

```python
SalePrice = np.expm1(predictions)
```

In [2]:
# ============================================================
# 2. Load Processed Data
# ============================================================

PROCESSED_PATH = Path(
    "/kaggle/input/notebooks/nomusashongwe/notebook3e3c1e3c92/processed"
)

X_train = pd.read_csv(PROCESSED_PATH / "X_train_fe.csv")
X_test = pd.read_csv(PROCESSED_PATH / "X_test_fe.csv")
y_train = pd.read_csv(PROCESSED_PATH / "y_train_log.csv")["SalePriceLog"]
test_ids = pd.read_csv(PROCESSED_PATH / "test_ids.csv")["Id"]
train_ids = pd.read_csv(PROCESSED_PATH / "train_ids.csv")["Id"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("train_ids shape:", train_ids.shape)
print("test_ids shape:", test_ids.shape)

X_train shape: (1460, 275)
X_test shape: (1459, 275)
y_train shape: (1460,)
train_ids shape: (1460,)
test_ids shape: (1459,)


In [3]:
X_train.head()

,Id,LotFrontage,LotArea,LotShape,Utilities,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,MiscVal,MoSold,YrSold,TotalHouseSF,TotalLivingSF,TotalFinishedSF,TotalPorchSF,TotalOutdoorSF,TotalBathrooms,TotalFullBaths,TotalHalfBaths,HouseAge,RemodAge,GarageAge,IsRemodeled,IsNewHouse,HasGarage,HasBasement,HasFireplace,HasPool,HasFence,HasAlley,...,Exterior1st_Plywood,Exterior1st_Stone,Exterior1st_Stucco,Exterior1st_VinylSd,Exterior1st_Wd Sdng,Exterior1st_WdShing,Exterior2nd_AsbShng,Exterior2nd_AsphShn,Exterior2nd_Brk Cmn,Exterior2nd_BrkFace,Exterior2nd_CBlock,Exterior2nd_CmentBd,Exterior2nd_HdBoard,Exterior2nd_ImStucc,Exterior2nd_MetalSd,Exterior2nd_Other,Exterior2nd_Plywood,Exterior2nd_Stone,Exterior2nd_Stucco,Exterior2nd_VinylSd,Exterior2nd_Wd Sdng,Exterior2nd_Wd Shng,MasVnrType_BrkCmn,MasVnrType_BrkFace,MasVnrType_None,MasVnrType_Stone,Foundation_BrkTil,Foundation_CBlock,Foundation_PConc,Foundation_Slab,Foundation_Stone,Foundation_Wood,Heating_Floor,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall,Electrical_FuseA,Electrical_FuseF,Electrical_FuseP,Electrical_Mix,Electrical_SBrkr,GarageType_2Types,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None,Fence_GdPrv,Fence_GdWo,Fence_MnPrv,Fence_MnWw,Fence_None,MiscFeature_Gar2,MiscFeature_None,MiscFeature_Othr,MiscFeature_Shed,MiscFeature_TenC,SaleType_COD,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,4.189655,9.042040,0.000000,3,0.000000,7,5,2003,2003,5.283204,1.609438,1.386294,4,3,0.693147,6,6.561031,0.693147,0.000000,5.017280,6.753438,5,1,6.753438,6.751101,0.000000,7.444833,1.000000,0.000000,2,1,3,0.693147,4,2.197225,7,0,0,"2,003.000000",2,2.000000,548.000000,3,3,2,0.000000,4.127134,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2,2008,7.850493,7.850493,7.790282,4.127134,4.127134,3.500000,3.000000,0.693147,5,5,5.000000,0,0,1,1,0,0,0,0,...,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
1,2,4.394449,9.169623,0.000000,3,0.000000,6,8,1976,1976,0.000000,1.386294,1.386294,4,3,1.609438,5,6.886532,0.693147,0.000000,5.652489,7.141245,5,1,7.141245,0.000000,0.000000,7.141245,0.000000,0.693147,2,0,3,0.693147,3,1.945910,7,1,3,"1,976.000000",2,2.000000,460.000000,3,3,2,5.700444,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5,2007,7.833996,7.833996,7.714677,0.000000,5.700444,2.500000,2.000000,0.693147,31,31,31.000000,0,0,1,1,1,0,0,0,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False
2,3,4.234107,9.328212,0.693147,3,0.000000,7,5,2001,2002,5.093750,1.609438,1.38629

In [4]:
y_train.head()

0   12.247699
1   12.109016
2   12.317171
3   11.849405
4   12.429220
Name: SalePriceLog, dtype: float64

---
## 2. Data Quality Check

Before training models, we confirm that the processed data contains:

- No missing values
- No infinite values
- Matching train-target dimensions
- Matching test-ID dimensions

In [5]:
# ============================================================
# Data Quality Checks
# ============================================================

print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

print("Infinite values in X_train:", np.isinf(X_train.select_dtypes(include=[np.number])).sum().sum())
print("Infinite values in X_test:", np.isinf(X_test.select_dtypes(include=[np.number])).sum().sum())

assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == test_ids.shape[0]

print("Data checks passed.")

Missing values in X_train: 0
Missing values in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0
Data checks passed.


---
## 3. Evaluation Metric

The Kaggle competition evaluates predictions using Root Mean Squared Logarithmic Error.

Because the target has already been transformed using `log1p(SalePrice)`, the validation score can be computed as RMSE on the log-transformed target.

Lower scores are better.

A score around:

- `0.13` is a solid baseline.
- `0.12` is strong.
- `0.11` or lower is highly competitive for this classic competition.

In [6]:
# ============================================================
# Evaluation Function
# ============================================================

def rmse_log(y_true, y_pred):
    """
    RMSE calculated on log-transformed SalePrice.
    This aligns with the RMSLE-style competition objective.
    """
    return np.sqrt(mean_squared_error(y_true, y_pred))


def cv_rmse(model, X, y, n_splits=5):
    """
    Cross-validated RMSE using KFold.
    Keeps pandas DataFrames where possible to preserve feature names.
    """
    kf = KFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    scores = -cross_val_score(
        model,
        X,
        y,
        scoring="neg_root_mean_squared_error",
        cv=kf,
        n_jobs=-1
    )

    return scores

---
## 4. Baseline Model Setup

This section trains a diverse set of regression models.

Linear and regularized models are wrapped in a pipeline with `RobustScaler`, because they are sensitive to feature scale and outliers.

Tree-based models are used without scaling because they are not distance-based.

In [7]:
# ============================================================
# Baseline Models
# Scaling is applied only to linear models
# ============================================================

models = {
    "Ridge": Pipeline([
        ("scaler", RobustScaler()),
        ("model", Ridge(alpha=10.0, random_state=RANDOM_STATE))
    ]),

    "Lasso": Pipeline([
        ("scaler", RobustScaler()),
        ("model", Lasso(
            alpha=0.0005,
            random_state=RANDOM_STATE,
            max_iter=70000
        ))
    ]),

    "ElasticNet": Pipeline([
        ("scaler", RobustScaler()),
        ("model", ElasticNet(
            alpha=0.0005,
            l1_ratio=0.9,
            random_state=RANDOM_STATE,
            max_iter=70000
        ))
    ]),

    "BayesianRidge": Pipeline([
        ("scaler", RobustScaler()),
        ("model", BayesianRidge())
    ]),

    "RandomForest": RandomForestRegressor(
        n_estimators=700,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=800,
        learning_rate=0.03,
        max_depth=3,
        min_samples_split=10,
        min_samples_leaf=3,
        subsample=0.8,
        random_state=RANDOM_STATE
    )
}

print("Baseline models created:", list(models.keys()))

Baseline models created: ['Ridge', 'Lasso', 'ElasticNet', 'BayesianRidge', 'RandomForest', 'GradientBoosting']


---
## 5. Optional Boosting Libraries

Kaggle usually has `xgboost`, `lightgbm`, and `catboost` available.

These models often perform very well on tabular regression problems.

The code below adds them only if the libraries are available in the environment.

In [8]:
# ============================================================
# Optional Boosting Models
# Tree-based boosting models are not scaled
# ============================================================

try:
    from xgboost import XGBRegressor

    models["XGBoost"] = XGBRegressor(
        n_estimators=1200,
        learning_rate=0.025,
        max_depth=3,
        min_child_weight=2,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0005,
        reg_lambda=1.0,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

except Exception as e:
    print("XGBoost not available:", e)


try:
    from lightgbm import LGBMRegressor

    models["LightGBM"] = LGBMRegressor(
        n_estimators=1400,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0005,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )

except Exception as e:
    print("LightGBM not available:", e)


try:
    from catboost import CatBoostRegressor

    models["CatBoost"] = CatBoostRegressor(
        iterations=1400,
        learning_rate=0.025,
        depth=5,
        l2_leaf_reg=3,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )

except Exception as e:
    print("CatBoost not available:", e)

print("All baseline models:", list(models.keys()))

All baseline models: ['Ridge', 'Lasso', 'ElasticNet', 'BayesianRidge', 'RandomForest', 'GradientBoosting', 'XGBoost', 'LightGBM', 'CatBoost']


---
## 6. Cross-Validation Model Comparison

Each model is evaluated using 5-fold cross-validation.

The score reported is RMSE on log-transformed `SalePrice`.

A lower score indicates better expected leaderboard performance.

In [9]:
# ============================================================
# Cross-Validation Comparison
# ============================================================

cv_results = []

for name, model in models.items():
    print(f"Training and validating: {name}")

    scores = cv_rmse(model, X_train, y_train, n_splits=5)

    cv_results.append({
        "model": name,
        "mean_rmse": scores.mean(),
        "std_rmse": scores.std(),
        "fold_scores": scores
    })

    print(f"{name}: {scores.mean():.6f} ± {scores.std():.6f}")

Training and validating: Ridge
Ridge: 0.129965 ± 0.021934
Training and validating: Lasso
Lasso: 0.125952 ± 0.022195
Training and validating: ElasticNet
ElasticNet: 0.126049 ± 0.022195
Training and validating: BayesianRidge
BayesianRidge: 0.130126 ± 0.021866
Training and validating: RandomForest
RandomForest: 0.140262 ± 0.017231
Training and validating: GradientBoosting
GradientBoosting: 0.127904 ± 0.020389
Training and validating: XGBoost
XGBoost: 0.126521 ± 0.019599
Training and validating: LightGBM
LightGBM: 0.133428 ± 0.017763
Training and validating: CatBoost
CatBoost: 0.124073 ± 0.019712


In [10]:
# ============================================================
# 8. Results Table
# ============================================================

results_df = pd.DataFrame(cv_results)
results_df = results_df.sort_values("mean_rmse").reset_index(drop=True)

results_df[["model", "mean_rmse", "std_rmse"]]

,model,mean_rmse,std_rmse
0,CatBoost,0.124073,0.019712
1,Lasso,0.125952,0.022195
2,ElasticNet,0.126049,0.022195
3,XGBoost,0.126521,0.019599
4,GradientBoosting,0.127904,0.020389
5,Ridge,0.129965,0.021934
6,BayesianRidge,0.130126,0.021866
7,LightGBM,0.133428,0.017763
8,RandomForest,0.140262,0.017231


---
## 7. Cross-Validation Interpretation

The baseline results show that the strongest early performers are CatBoost, Lasso, ElasticNet, XGBoost, and Gradient Boosting.

Several important patterns are visible:

1. CatBoost performs best among the baseline models, suggesting that boosted tree models can capture important nonlinear relationships in the feature-engineered dataset.

2. Lasso and ElasticNet remain highly competitive, which indicates that the feature engineering process successfully created clean and informative predictors.

3. Random Forest underperforms compared with boosting and regularized linear models. This suggests that bagging-based trees are less effective for this dataset than boosting-based models.

4. The small gap between the strongest models suggests that no single model dominates completely.

The next step is to evaluate stronger tuned versions of the most promising model families.

In [11]:
best_model_name = results_df.loc[0, "model"]
best_score = results_df.loc[0, "mean_rmse"]

print("Best baseline model:", best_model_name)
print("Best baseline CV RMSE:", round(best_score, 6))

Best baseline model: CatBoost
Best baseline CV RMSE: 0.124073


---
## 8. Tuned Candidate Models

This section defines tuned models that are typically strong for the Ames Housing competition.

The focus is on:

- Regularized linear models
- Gradient boosting models
- Tree ensembles

These models are later used for blending.

In [12]:
# ============================================================
# Tuned Candidate Models
# Scaling is applied only to linear models
# ============================================================

tuned_models = {
    "Tuned_Ridge": Pipeline([
        ("scaler", RobustScaler()),
        ("model", Ridge(alpha=13.0, random_state=RANDOM_STATE))
    ]),

    "Tuned_Lasso": Pipeline([
        ("scaler", RobustScaler()),
        ("model", Lasso(
            alpha=0.00045,
            random_state=RANDOM_STATE,
            max_iter=70000
        ))
    ]),

    "Tuned_ElasticNet": Pipeline([
        ("scaler", RobustScaler()),
        ("model", ElasticNet(
            alpha=0.00045,
            l1_ratio=0.85,
            random_state=RANDOM_STATE,
            max_iter=70000
        ))
    ]),

    "Tuned_BayesianRidge": Pipeline([
        ("scaler", RobustScaler()),
        ("model", BayesianRidge())
    ]),

    "Tuned_GradientBoosting": GradientBoostingRegressor(
        n_estimators=1200,
        learning_rate=0.025,
        max_depth=3,
        min_samples_split=10,
        min_samples_leaf=3,
        max_features="sqrt",
        subsample=0.85,
        random_state=RANDOM_STATE
    )
}

print("Base tuned models:", list(tuned_models.keys()))

Base tuned models: ['Tuned_Ridge', 'Tuned_Lasso', 'Tuned_ElasticNet', 'Tuned_BayesianRidge', 'Tuned_GradientBoosting']


In [13]:
# ============================================================
# Add Tuned Optional Boosting Models
# ============================================================

try:
    tuned_models["Tuned_XGBoost"] = XGBRegressor(
        n_estimators=1800,
        learning_rate=0.018,
        max_depth=3,
        min_child_weight=2,
        subsample=0.82,
        colsample_bytree=0.82,
        gamma=0,
        reg_alpha=0.0005,
        reg_lambda=1.2,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
except NameError:
    pass


try:
    tuned_models["Tuned_LightGBM"] = LGBMRegressor(
        n_estimators=2000,
        learning_rate=0.015,
        num_leaves=20,
        max_depth=-1,
        min_child_samples=18,
        subsample=0.82,
        colsample_bytree=0.82,
        reg_alpha=0.0005,
        reg_lambda=1.2,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )
except NameError:
    pass


try:
    tuned_models["Tuned_CatBoost"] = CatBoostRegressor(
        iterations=2000,
        learning_rate=0.018,
        depth=5,
        l2_leaf_reg=4,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=False
    )
except NameError:
    pass

print("Tuned models:", list(tuned_models.keys()))

Tuned models: ['Tuned_Ridge', 'Tuned_Lasso', 'Tuned_ElasticNet', 'Tuned_BayesianRidge', 'Tuned_GradientBoosting', 'Tuned_XGBoost', 'Tuned_LightGBM', 'Tuned_CatBoost']


---
## 9. Tuned Model Evaluation

The tuned models are evaluated using the same 5-fold cross-validation framework.

This allows fair comparison against the baseline models.

In [15]:
# ============================================================
# Evaluate Tuned Models
# ============================================================

tuned_results = []

for name, model in tuned_models.items():
    print(f"Training and validating: {name}")

    scores = cv_rmse(model, X_train, y_train, n_splits=5)

    tuned_results.append({
        "model": name,
        "mean_rmse": scores.mean(),
        "std_rmse": scores.std(),
        "fold_scores": scores
    })

    print(f"{name}: {scores.mean():.6f} ± {scores.std():.6f}")

Training and validating: Tuned_Ridge
Tuned_Ridge: 0.129896 ± 0.021893
Training and validating: Tuned_Lasso
Tuned_Lasso: 0.126002 ± 0.022213
Training and validating: Tuned_ElasticNet
Tuned_ElasticNet: 0.126173 ± 0.022269
Training and validating: Tuned_BayesianRidge
Tuned_BayesianRidge: 0.130126 ± 0.021866
Training and validating: Tuned_GradientBoosting
Tuned_GradientBoosting: 0.124572 ± 0.021624
Training and validating: Tuned_XGBoost
Tuned_XGBoost: 0.126565 ± 0.018915
Training and validating: Tuned_LightGBM
Tuned_LightGBM: 0.131631 ± 0.017509
Training and validating: Tuned_CatBoost
Tuned_CatBoost: 0.124062 ± 0.019563


In [16]:
tuned_results_df = pd.DataFrame(tuned_results)
tuned_results_df = tuned_results_df.sort_values("mean_rmse").reset_index(drop=True)

tuned_results_df[["model", "mean_rmse", "std_rmse"]]

,model,mean_rmse,std_rmse
0,Tuned_CatBoost,0.124062,0.019563
1,Tuned_GradientBoosting,0.124572,0.021624
2,Tuned_Lasso,0.126002,0.022213
3,Tuned_ElasticNet,0.126173,0.022269
4,Tuned_XGBoost,0.126565,0.018915
5,Tuned_Ridge,0.129896,0.021893
6,Tuned_BayesianRidge,0.130126,0.021866
7,Tuned_LightGBM,0.131631,0.017509


---
## 10. Combined Model Leaderboard

This section combines baseline and tuned model results into one leaderboard.

The best-performing models from this table will be selected for final blending.

In [17]:
# ============================================================
# Combined Leaderboard
# ============================================================

combined_results_df = pd.concat(
    [
        results_df[["model", "mean_rmse", "std_rmse"]].assign(stage="baseline"),
        tuned_results_df[["model", "mean_rmse", "std_rmse"]].assign(stage="tuned")
    ],
    axis=0,
    ignore_index=True
)

combined_results_df = combined_results_df.sort_values("mean_rmse").reset_index(drop=True)

combined_results_df

,model,mean_rmse,std_rmse,stage
0,Tuned_CatBoost,0.124062,0.019563,tuned
1,CatBoost,0.124073,0.019712,baseline
2,Tuned_GradientBoosting,0.124572,0.021624,tuned
3,Lasso,0.125952,0.022195,baseline
4,Tuned_Lasso,0.126002,0.022213,tuned
5,ElasticNet,0.126049,0.022195,baseline
6,Tuned_ElasticNet,0.126173,0.022269,tuned
7,XGBoost,0.126521,0.019599,baseline
8,Tuned_XGBoost,0.126565,0.018915,tuned
9,GradientBoosting,0.127904,0.020389,baseline


---
## 11. Select Final Models for Blending

Final models should be selected from the tuned candidate set, not from a mixed baseline-and-tuned leaderboard.

The final blend focuses on model diversity:

- Regularized linear models capture stable linear relationships.
- Gradient boosting captures nonlinear effects and interactions.
- CatBoost captures complex feature patterns and performs best in cross-validation.
- XGBoost adds another boosted-tree perspective.

Weak or redundant models are excluded from the final blend.

In [31]:
# ============================================================
# 12. Final Blend Model Selection
# ============================================================

top_models = tuned_results_df.sort_values("mean_rmse").head(5)

final_model_names = top_models["model"].tolist()

final_models = {
    name: tuned_models[name]
    for name in final_model_names
}

print("Final models selected:")
for name in final_models:
    print("-", name)

top_models

Final models selected:
- Tuned_CatBoost
- Tuned_GradientBoosting
- Tuned_Lasso
- Tuned_ElasticNet
- Tuned_XGBoost


,model,mean_rmse,std_rmse,fold_scores
0,Tuned_CatBoost,0.124062,0.019563,"[0.1281930052839959, 0.10877386374015544, 0.15..."
1,Tuned_GradientBoosting,0.124572,0.021624,"[0.12524767898058448, 0.11112893138708047, 0.1..."
2,Tuned_Lasso,0.126002,0.022213,"[0.12194331441326929, 0.11615492321006653, 0.1..."
3,Tuned_ElasticNet,0.126173,0.022269,"[0.12135509784033711, 0.11663199883382974, 0.1..."
4,Tuned_XGBoost,0.126565,0.018915,"[0.12815688770214576, 0.11480877950019783, 0.1..."


---
## 12. Fit Final Models on Full Training Data

After cross-validation, the selected models are trained on the full feature-engineered training dataset.

These models will be used to predict the Kaggle test set.

In [20]:
# ============================================================
# Fit Final Models
# ============================================================

fitted_models = {}

for name, model in final_models.items():
    print(f"Fitting {name} on full training data...")
    model.fit(X_train, y_train)
    fitted_models[name] = model

print("All final models fitted.")

Fitting Tuned_CatBoost on full training data...
Fitting Tuned_GradientBoosting on full training data...
Fitting Tuned_XGBoost on full training data...
Fitting Tuned_Lasso on full training data...
Fitting Tuned_ElasticNet on full training data...
All final models fitted.


---
## 13. Generate Individual Model Predictions

Each model predicts log-transformed sale prices.

The predictions are converted back to the original sale price scale using:

```python
np.expm1(predictions)
```

Before blending on the original scale, predictions are clipped to prevent unrealistic negative values.

In [21]:
# ============================================================
# Individual Test Predictions
# ============================================================

test_predictions_log = {}
test_predictions = {}

for name, model in fitted_models.items():
    pred_log = model.predict(X_test)
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)

    test_predictions_log[name] = pred_log
    test_predictions[name] = pred

prediction_df = pd.DataFrame(test_predictions)
prediction_df.head()

,Tuned_CatBoost,Tuned_GradientBoosting,Tuned_XGBoost,Tuned_Lasso,Tuned_ElasticNet
0,"122,813.098643","125,577.147508","121,758.679688","115,483.418895","115,921.946800"
1,"159,545.041777","160,864.066367","160,272.531250","159,049.840346","159,077.559625"
2,"173,934.613710","177,619.515985","177,960.250000","181,286.297519","180,953.900229"
3,"191,922.887566","193,212.999464","194,414.234375","195,086.169361","194,860.989621"
4,"187,832.287000","184,322.527022","185,400.250000","197,536.875745","198,460.365520"


In [22]:
prediction_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Tuned_CatBoost,"1,459.000000","176,236.258917","74,343.800606","37,403.264803","127,111.685525","155,744.800652","206,431.094739","494,556.010339"
Tuned_GradientBoosting,"1,459.000000","175,578.943838","74,556.370220","38,732.318346","126,456.613070","155,036.250430","205,698.134600","605,619.506197"
Tuned_XGBoost,"1,459.000000","176,367.921875","74,478.828125","36,225.500000","127,246.351562","156,589.375000","206,518.609375","538,340.312500"
Tuned_Lasso,"1,459.000000","176,284.745296","74,059.612494","39,571.076629","125,533.657049","157,343.079620","208,992.576842","648,938.370091"
Tuned_ElasticNet,"1,459.000000","176,299.437530","74,277.747908","39,104.502776","125,454.581900","156,855.481446","209,414.820184","651,096.558269"


---
## 14. Blended Prediction

The final prediction is a weighted blend of the selected tuned models.

Weights are computed automatically using inverse cross-validation RMSE. Better models receive higher weight, ensuring a performance-driven blend.

A power term is applied to slightly increase the influence of the strongest models while still preserving model diversity.

In [32]:
# ============================================================
# Power-Weighted RMSE Blend
# ============================================================

available_models = prediction_df.columns.tolist()

score_lookup = dict(
    zip(tuned_results_df["model"], tuned_results_df["mean_rmse"])
)

power = 2

weights = {
    model_name: (1 / score_lookup[model_name]) ** power
    for model_name in available_models
    if model_name in score_lookup
}

weight_sum = sum(weights.values())

weights = {
    model_name: weight / weight_sum
    for model_name, weight in weights.items()
}

weights

{'Tuned_CatBoost': 0.20454480107612288,
 'Tuned_GradientBoosting': 0.20287293330827164,
 'Tuned_XGBoost': 0.19653236906893645,
 'Tuned_Lasso': 0.1982923206229068,
 'Tuned_ElasticNet': 0.1977575759237623}

In [24]:
# ============================================================
# Blend on Original Price Scale
# ============================================================

final_predictions = np.zeros(len(X_test))

for model_name, weight in weights.items():
    final_predictions += weight * prediction_df[model_name].values

final_predictions = np.clip(final_predictions, 0, None)

final_predictions[:10]

array([120330.59550238, 159763.38588284, 178335.13200342, 193891.01406369,
       190689.50786125, 172098.64103043, 177476.31185923, 163718.48086382,
       186509.06924753, 122340.98929358])

In [25]:
pd.Series(final_predictions).describe()

count     1,459.000000
mean    176,152.177311
std      73,909.594444
min      39,292.036302
25%     126,758.947161
50%     156,238.527124
75%     208,422.267117
max     504,330.487729
dtype: float64

---
## 15. Create Kaggle Submission

The submission file must contain exactly two columns:

- `Id`
- `SalePrice`

The `Id` values must come from the test set.

In [26]:
# ============================================================
# Create Submission File
# ============================================================

submission = pd.DataFrame({
    "Id": test_ids,
    "SalePrice": final_predictions
})

submission.head()

,Id,SalePrice
0,1461,"120,330.595502"
1,1462,"159,763.385883"
2,1463,"178,335.132003"
3,1464,"193,891.014064"
4,1465,"190,689.507861"


In [27]:
submission.shape

(1459, 2)

In [28]:
submission.to_csv("/kaggle/working/submission.csv", index=False)

print("Submission file saved to /kaggle/working/submission.csv")

Submission file saved to /kaggle/working/submission.csv


---
## 16. Save Modeling Artifacts

The model comparison tables and individual prediction files are saved for review.

This helps document which models performed best and how the final prediction was created.

In [29]:
# ============================================================
# 17. Save Artifacts
# ============================================================

ARTIFACT_DIR = Path("/kaggle/working/modeling_outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

combined_results_df.to_csv(ARTIFACT_DIR / "model_cv_results.csv", index=False)
prediction_df.to_csv(ARTIFACT_DIR / "individual_model_predictions.csv", index=False)

pd.DataFrame({
    "model": list(weights.keys()),
    "weight": list(weights.values())
}).to_csv(ARTIFACT_DIR / "blend_weights.csv", index=False)

print("Modeling artifacts saved to:", ARTIFACT_DIR)

Modeling artifacts saved to: /kaggle/working/modeling_outputs


In [30]:
os.listdir("/kaggle/working")

['submission.csv', '.virtual_documents', 'catboost_info', 'modeling_outputs']

---
## 17. Modeling Summary

This notebook trained, evaluated, tuned, and blended multiple regression models for the Ames Housing price prediction task.

Key steps completed:

1. Loaded the feature-engineered datasets from Notebook 02.
2. Verified that the processed train and test matrices contained no missing or infinite values.
3. Defined RMSE on log-transformed `SalePrice` as the validation metric.
4. Evaluated baseline linear, tree, and boosting models using 5-fold cross-validation.
5. Removed weak or unstable models from the final modeling strategy.
6. Tuned the strongest model families, including regularized linear models and gradient boosting models.
7. Compared baseline and tuned model performance in a combined leaderboard.
8. Selected final models from the tuned candidate set.
9. Fitted the selected final models on the full training dataset.
10. Generated individual model predictions for the Kaggle test set.
11. Created a performance-based weighted blend using inverse cross-validation RMSE.
12. Saved a Kaggle-ready `submission.csv`.
13. Saved modeling artifacts, including model scores, individual predictions, and blend weights.

The next step is to submit `submission.csv` to Kaggle and compare the public leaderboard score against the cross-validation estimates.